In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# GRPO: a baseline without a critic

## Where this session fits

> **The limitation we start from: PPO's advantage needs a learned critic, a second large network that must score every prefix of a long response, and whose mistakes become bias**

- **The question:** can the baseline come from the episodes themselves, with no learned critic?
- **Position: Part C (language models), session 2 of 2**
    - the last session made a language model a policy and showed one training step
    - this session builds a critic-free method from Part B's PPO
    - after this session: the Recap, then the Advanced course, with PPO and GRPO in depth, reward models, RLHF and reasoning models
- **The three slots** of the plan from B1 (theorem): the parts of the gradient formula that the methods change
    1. **slot 1, the weight** $\advseq$ on each gradient term
    2. **slot 2, how far** one update moves the policy (PPO's clipped ratio, B4 (step size))
    3. **slot 3, where the baseline comes from** (inside slot 1) **← this session**
- **Series goal:** choose good actions by trial and error, from small tables to robots and language models

**By the end you can**

1. say how a group of episodes from one start state can serve as a baseline
2. compute a group-relative advantage from the rewards of a group
3. read the GRPO objective as PPO's clipped objective with a different advantage
4. say what the group baseline gives and what it costs, compared with a critic
5. place GRPO in the finished slot table


**Recall: PPO's clipped objective from B4**

> **GRPO keeps this objective and changes only the advantage $\advseq_\tt$.**

$$
J_{\text{clip}}(\theta) = \E_\tt \Bigl[\, \min{} \bigl( r_\tt(\theta)\, \advseq_\tt, \;\; \text{clip}\bigl(r_\tt(\theta),\, 1 - \epsilon_{\text{clip}},\, 1 + \epsilon_{\text{clip}}\bigr)\, \advseq_\tt \bigr) \,\Bigr]
$$
- the probability ratio: $r_\tt(\theta) = \pi_\theta(\actseq_\tt \mid \stateseq_\tt) \, / \, \pi_{\theta_{\text{old}}}(\actseq_\tt \mid \stateseq_\tt)$
- PPO took $\advseq_\tt$ from a critic; this session takes it from a group

## The idea: the episodes themselves are the baseline

**Why this section?** Part B showed that a baseline only has to be a number that does not depend on the action (B2 weight). A critic is one source. A group of samples is another.

> **If we can restart from the same state $G$ times, the average return of those $G$ episodes estimates the value of that state: a baseline with nothing to train.**

- the baseline $b(\state)$ must not depend on the action taken (B2 (weight))
    - a critic learns $\statevalfun(\state)$ from many episodes and many states
- a **group**: sample $G \geq 2$ episodes from the *same* start state, with the policy as it is now
    - $G$ is the group size, as in the GRPO papers
    - in Part B and Sutton-Barto $G$ is the return: the context tells which
    - their **average return** is a **Monte Carlo** estimate of the value of that state (Part A)
        - "Monte Carlo" here means two things at once: estimate by averaging samples, and use each episode's complete return, with no bootstrapping
    - a larger $G$ makes the average less noisy: the noise of an average of $G$ returns falls like $1/\sqrt{G}$
        - the price is $G$ times the generation
    - use it as the baseline for every episode of the group: there is no network to train
    - it is the value of the *start* state, used at every later step too: that is why there is no per-step credit
- it needs two things
    - a start state we can return to: a simulator reset, or a prompt
    - one return per episode, known at the end
- the baseline of REINFORCE (B2 weight) pooled *all* past episodes in a moving average
    - the group baseline uses the *same* start state, with the *current* policy

**One start state, eight episodes** (synthetic returns)

<img src="images/c2_group_baseline.png" width=90%>

- an episode's **centered return** is its return minus the group mean (the next section also divides by the group standard deviation to get the advantage)
- subtracting the group mean does two jobs
    - it lowers the variance, and a baseline that ignores the episode's own action leaves the expected gradient unchanged (B2)
    - it gives each episode a sign
        - above the mean: raise the probability of its actions
        - below the mean: lower it
- caveat: the group mean contains the episode's own return, so it is not exactly independent of it
    - "What the group baseline gives and costs", later in this session, quantifies this
    - a larger $G$ shrinks the effect
- B2 (weight) drew the contrast: a moving average of past episodes (REINFORCE) against a group mean (this slide)

**Why a language model suits it**

- the **prompt is the start state**, and a language model can restart from it for free: sample again
- the reward is one number per completion, known at the end
- the critic would be a second large network that scores every prefix (the last session, and B3 (critic))
- **GRPO** (group relative policy optimization) was introduced for language models, and the rest of this session is its details
- outside language models the idea needs a resettable start state and one return per episode, so it fits simulators better than real robots


## The group-relative advantage

**Why this section?** The one new formula. Everything else in GRPO is PPO.

> **A completion's advantage is its reward relative to its group: its reward minus the group mean, divided by the group standard deviation.**

- completion $\y_i$ has a **reward** $\rew_i = \rew(\x, \y_i)$ (a **completion** is the model's response to the prompt $\x$, as in C1)
    - a single number for the whole completion, as in C1
    - it is the return of the episode, since nothing else is rewarded
    - written $r_i$ in the GRPO papers, and here
    - the reward is often a simple rule (right or wrong, plus format checks), not a calibrated reward model
    - only the order and spacing of the rewards within a group matter
        - adding a constant to all of them leaves the advantages unchanged
        - so does multiplying them by a positive number
- its advantage is its reward relative to the group

$$
\hat{\advseq}_i = \frac{\rew_i - \operatorname{mean}(\rew_1, \ldots, \rew_G)}{\operatorname{std}(\rew_1, \ldots, \rew_G)}
$$

- the standard deviation is the sample standard deviation (divisor $G - 1$)
    - if all $G$ rewards are equal it is $0$: the advantage is then set to $0$ (implementations add a tiny constant)
- the same advantage $\hat{\advseq}_i$ is used for **every token** of completion $i$
    - the reward is one number per completion, and there is no critic to assign credit to each step

**The group advantage by numbers**

- one prompt, $G = 4$ completions, returns $(1, 0, 0, 0)$
    - mean $0.25$
    - sample standard deviation $0.5$ (divisor $G - 1$)
- advantages
    - the success: $(1 - 0.25)/0.5 = 1.5$
    - each of the three failures: $(0 - 0.25)/0.5 = -0.5$
- dividing by the standard deviation rescales each prompt's advantages to a common size
    - it also gives more weight to prompts whose rewards nearly all agree: a known criticism
        - a small standard deviation inflates the advantages, as with one success among eight
    - at $G = 2$ every advantage is $\pm 0.71$ whatever the gap between the two returns

**Three prompts, one picture** (synthetic rewards, $G = 4$)

<img src="images/c2_group_advantage.png" width=95%>

- prompt 1: one success among four is a rare event, so it gets a large positive advantage and the failures small negative ones
- prompt 2: two and two: symmetric advantages
- prompt 3: all four succeed, so every advantage is $0$: **no policy-gradient signal from this prompt**
    - the KL term still acts
    - the same happens if all four fail
- the advantage signal comes only from prompts whose completions *differ* in reward


## The GRPO objective: PPO's clipped objective, a new advantage

**Why this section?** Slot 2 is not new: we keep the clipped ratio of B4 (step size). The advantage (slots 3 and 1) is what changes.

> **The GRPO objective is PPO's clipped objective with the group-relative advantage, plus a KL penalty to a frozen reference model.**

$$
\frac{1}{G} \sum_{i=1}^{G} \frac{1}{|\y_i|} \sum_{\tt} \Bigl[ \min{} \bigl( r_{i,\tt}(\theta)\, \hat{\advseq}_i, \;\; \text{clip}\bigl(r_{i,\tt}(\theta),\, 1 - \epsilon_{\text{clip}},\, 1 + \epsilon_{\text{clip}}\bigr)\, \hat{\advseq}_i \bigr) \; - \; \beta \, \KL_{i,\tt} \Bigr]
$$

- $r_{i,\tt}(\theta)$ is the probability ratio of token $\tt$ of completion $i$, new policy over old (B4 (step size))
- $|\y_i|$ is the number of tokens of completion $i$, and $\tt$ runs over them: the objective is averaged over tokens, then over completions
- $\KL_{i,\tt}$ is the per-token estimate of the KL penalty (two sub-slides on), with weight $\beta$

**Same as PPO, and not the same**

- **the same** as PPO's clipped objective (B4): the ratio $r_{i,\tt}(\theta)$, the clip and the $\min{}$
- **what differs**
    - the **advantage**: one per completion, shared by all its tokens (slots 3 and 1)
    - the **averaging**: over each completion's tokens first, then over the group
    - a **KL term to a frozen reference model**, which PPO's clipped objective does not have
        - B4's KL-penalty option is a single term that anchors to the *old* policy
        - here the anchor is the *reference*, estimated on each sampled token and averaged inside the sum
    - there is **no value loss**, because there is no critic

**The KL term: a loss term here, a reward in PPO**

- the $\KL$ term, $\KL(\pi_\theta \,\|\, \pi_{\text{ref}})$, anchors the policy to the frozen **reference model** $\pi_{\text{ref}}$ of the last session, with weight $\beta$
    - for a language model, the reference is the model we started from
    - it is estimated on the sampled tokens: with $d = \log \pi_{\text{ref}} - \log \pi_\theta$ at the sampled token, the estimate is $e^{d} - d - 1$ (which is never negative)
- in PPO (the last session) the KL was a **per-token reward**, which the critic and GAE handle
- GRPO has one advantage per completion and no critic, so it has no per-step reward to put the KL in
    - the GRPO paper puts it in the **loss** instead, which keeps the advantage a pure group comparison
    - some recipes set $\beta$ to zero: the Advanced course returns to this choice
- notation: $r_{i,\tt}$ is a *ratio* here
    - $r_i$, with one index, is the reward of completion $i$
    - $r_{i,\tt}$, with two, is the ratio
- the objective is maximized; code minimizes its negative

**PPO versus GRPO: what changes**

<table>
<thead><tr><th nowrap><div align="left"></div></th><th nowrap><div align="left">PPO (B4 (step size))</div></th><th nowrap><div align="left">GRPO</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">baseline source (slot 3)</div></td><td nowrap><div align="left">learned critic $\statevalfun_\phi$</div></td><td nowrap><div align="left"><b>mean of a group of completions</b></div></td></tr>
<tr><td nowrap><div align="left">weight (slot 1)</div></td><td nowrap><div align="left">GAE advantage, one per step</div></td><td nowrap><div align="left"><b>group-relative advantage, one per completion</b></div></td></tr>
<tr><td nowrap><div align="left">how far (slot 2)</div></td><td nowrap><div align="left">clipped ratio, $K$ epochs</div></td><td nowrap><div align="left">same</div></td></tr>
<tr><td nowrap><div align="left">KL to a reference model</div></td><td nowrap><div align="left">a per-token reward (RLHF-style)</div></td><td nowrap><div align="left">a loss term, usually</div></td></tr>
<tr><td nowrap><div align="left">averaging</div></td><td nowrap><div align="left">over the tokens of the batch</div></td><td nowrap><div align="left">over each completion's tokens, then over the group</div></td></tr>
<tr><td nowrap><div align="left">networks</div></td><td nowrap><div align="left">actor and critic (and a reference model for language models)</div></td><td nowrap><div align="left">actor and a reference model</div></td></tr>
<tr><td nowrap><div align="left">extra cost</div></td><td nowrap><div align="left">train the critic</div></td><td nowrap><div align="left">sample $G$ completions per prompt</div></td></tr>
</tbody></table>

- the old actor is only a snapshot (often just its stored log-probabilities) in both methods


**The models of a GRPO step**

<img src="images/c2_models.png" width=90%>

- compare with the PPO step of C1: the critic is gone
- the **old actor** is still there: the snapshot that generated the group, kept as stored log-probabilities, for the ratio
    - it matters only when a batch is reused for several updates, as in PPO
    - the objective is the general case, with $K$ updates per batch
    - with one update per batch the ratio is exactly $1$, and the clip never acts
    - the DeepSeekMath paper's GRPO training reports one update per exploration stage
- the baseline comes from the group, so the extra cost is generation: $G$ completions per prompt
- the reference is still needed for the KL term


## GRPO in one loop

**Why this section?** Compare with the PPO loop of B4 (step size): the critic and the GAE are gone, and the group appears.

> **GRPO in one loop: PPO's loop without the critic and GAE, with a group of completions per prompt.**

<img src="images/c2_grpo_loop.png" width=95%>

- for each iteration, for each prompt: sample $G$ completions from the old policy, score them, turn the rewards into advantages
- then $K$ epochs of PPO's clipped step on that batch, plus the KL penalty
- the detailed loop, with the log-probabilities and the KL estimate, is in the aside at the end
- compare with the REINFORCE loop of B2 (weight): the same steps
    - only the baseline changes: a group mean, not a moving average of past episodes

**What is computed once, and what changes every epoch**

- once per iteration, with no gradient
    - the old policy's log-probabilities of the sampled tokens
    - the reference model's log-probabilities of the same tokens
    - the group advantages, copied to every token of the completion
- every epoch
    - the ratio of the current policy to the old one, then the clipped objective
    - the KL penalty to the reference
    - one update of $\theta$
- all-equal rewards give advantage $0$: those completions add nothing to the policy gradient


## What the group baseline gives, and what it costs

**Why this section?** There is no free baseline. Each source trades something, which is the bias and variance thread again.

> **The group baseline removes the critic, and pays with samples, a noisier baseline, no per-step credit, and no signal when the rewards agree.**

<table>
<thead><tr><th nowrap><div align="left">It gives</div></th><th nowrap><div align="left">It costs</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left"><b>no critic:</b> no second network to train, store and run</div></td><td nowrap><div align="left"><b>generation:</b> $G$ completions per prompt, every iteration</div></td></tr>
<tr><td nowrap><div align="left">no critic-approximation bias: the baseline comes from the policy's own samples</div></td><td nowrap><div align="left"><b>noise and a shrink:</b> only $G$ samples (so likely noisier than a critic), and the mean includes the completion itself</div></td></tr>
<tr><td nowrap><div align="left">one reward per completion is enough</div></td><td nowrap><div align="left"><b>no per-step credit:</b> every token shares one weight</div></td></tr>
<tr><td nowrap><div align="left">simple: one formula on top of PPO</div></td><td nowrap><div align="left"><b>no signal</b> when all rewards in a group agree</div></td></tr>
</tbody></table>

- the compute moves from a trained critic to generation
    - a group of $8$ costs about eight times the sampling per prompt
    - so in practice a batch uses fewer prompts to keep the total fixed

**The shrink, in numbers**

> **The shrink: the group mean includes the completion's own reward, so the advantage is a uniformly smaller version of the "leave-one-out" advantage.**

- two ways to form the baseline for completion $i$
    - the **group mean**, which includes $r_i$ itself
    - the **leave-one-out mean**, of the *other* $G - 1$ completions: independent of $r_i$
        - independence of $r_i$ is the condition of the zero-mean score lemma (B1): no bias
- the difference between the reward and the group mean is exactly $\frac{G-1}{G}$ times the difference between the reward and the leave-one-out mean
- example: $G = 4$, rewards $(1, 0, 0, 0)$, completion $1$
    - group mean $0.25$, so $1 - 0.25 = 0.75$
    - leave-one-out mean $0$, so $1 - 0 = 1$
    - $0.75 = \frac{3}{4} \times 1$
- so the advantage (before dividing by the standard deviation) is **shrunk** by the factor $\frac{G-1}{G}$
    - $0.5$ for $G = 2$, $0.875$ for $G = 8$
    - a rescaling and not a reversal: a smaller step, in the same direction
    - using the leave-one-out mean would remove it

**Two more costs of a small group**

- dividing by the standard deviation makes each weight depend on all the samples, including its own: a separate, smaller distortion
- the noise: the baseline is estimated from only $G$ samples (for example $8$)
    - PPO's critic generalizes across many episodes and states, and has its own bias

**When it fits**

- **it fits** when the reward is one number per completion
    - a verifier is the cleanest case (RLVR, C1)
    - a reward model or judge also works
    - a long response with one wrong step punishes all of its tokens: the price of whole-completion credit
    - a critic could in principle assign blame by step, though in practice critics are often poor at it
- **it is a poor fit** when we need credit inside a long episode
- **it needs completions that differ:** all-success and all-failure groups waste their samples
    - supervised fine-tuning first (the last session) makes mixed groups more likely


**Known weak spots: where the Advanced course picks up**

- this session showed the original GRPO; later work refines it
- each weak spot sits in one of the slots or in the sampling
    - **weight (slot 1):** the division by the standard deviation, and averaging per completion before averaging over the group
    - **how far (slot 2):** the clip range is the same size above and below $1$
    - **sampling:** a group whose rewards all agree gives no signal
    - **the KL term:** some recipes drop it
    - **credit:** one weight for every token of a completion
- recipes such as DAPO (a published refinement of GRPO) change several of these
    - the Advanced course covers them after PPO in depth


## The finished slot table

**Why this section?** One formula, three slots. GRPO fills the last open entry of Part B's table.

<table>
<thead><tr><th nowrap><div align="left">Method</div></th><th nowrap><div align="left">Slot 1: weight $\advseq$</div></th><th nowrap><div align="left">Slot 2: how far</div></th><th nowrap><div align="left">Slot 3: baseline source</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">raw theorem (B1)</div></td><td nowrap><div align="left">return, no baseline</div></td><td nowrap><div align="left">plain step</div></td><td nowrap><div align="left">none</div></td></tr>
<tr><td nowrap><div align="left">REINFORCE (B2)</div></td><td nowrap><div align="left">return minus baseline</div></td><td nowrap><div align="left">plain step</div></td><td nowrap><div align="left">moving average of returns</div></td></tr>
<tr><td nowrap><div align="left">actor-critic (B3)</div></td><td nowrap><div align="left">TD error or GAE</div></td><td nowrap><div align="left">plain step</div></td><td nowrap><div align="left">learned critic</div></td></tr>
<tr><td nowrap><div align="left">PPO (B4)</div></td><td nowrap><div align="left">GAE with a learned critic</div></td><td nowrap><div align="left">clipped ratio, $K$ epochs per batch</div></td><td nowrap><div align="left">learned critic</div></td></tr>
<tr><td nowrap><div align="left"><b>GRPO (C2)</b></div></td><td nowrap><div align="left"><b>group-relative advantage</b></div></td><td nowrap><div align="left">clipped ratio, $K$ epochs per batch</div></td><td nowrap><div align="left"><b>mean of a group of completions</b></div></td></tr>
</tbody></table>

- all rows share the theorem: a score function $\nabla_\theta \log \pi_\theta$ times a weight, averaged over sampled episodes
- the slots are separate choices: GRPO keeps PPO's slot 2 and changes slot 3, and slot 1 with it
- the slots are the only places the methods differ, so a new method is a new choice in one slot


## Aside: the GRPO loop in detail

- **models:** actor $\pi_\theta$, old actor $\pi_{\text{old}}$ (a snapshot), frozen reference $\pi_{\text{ref}}$
    - no critic
- for each iteration
    - $\pi_{\text{old}} \leftarrow \pi_\theta$ (once per iteration)
    - for each prompt $\x$ in the batch
        - sample a group of $G$ completions from $\pi_{\text{old}}$
        - one reward per completion: its reward $\rew_i$
        - advantage $\hat{\advseq}_i = (\rew_i - \text{mean}) / (\text{std} + 10^{-8})$
            - mean and standard deviation are over the $G$ rewards of *this prompt*
            - standard deviation with divisor $G - 1$
            - all-equal rewards give advantage $0$
        - copy $\hat{\advseq}_i$ to every token of completion $i$
        - compute once, with no gradient: $\log \pi_{\text{old}}$ and $\log \pi_{\text{ref}}$ of the sampled tokens
    - for epoch $= 1, \ldots, K$
        - per token: ratio $r = \exp(\log \pi_\theta - \log \pi_{\text{old}})$ (a ratio here, not a reward)
        - $d = \log \pi_{\text{ref}} - \log \pi_\theta$ (only $\pi_\theta$ carries a gradient)
        - KL estimate $e^{d} - d - 1$ (non-negative)
            - it estimates $\KL(\pi_\theta \,\|\, \pi_{\text{ref}})$
            - exactly so only for the first epoch, when the tokens come from $\pi_\theta$
        - loss: minus the mean of $\min{}\bigl(r\,\hat{\advseq}, \; \text{clip}(r, 1 - \epsilon_{\text{clip}}, 1 + \epsilon_{\text{clip}})\,\hat{\advseq}\bigr)$, plus $\beta$ times the mean KL estimate
            - means over the tokens of each completion, then over completions
            - a design choice: other variants average over all tokens
        - update $\theta$
- how best to differentiate the KL term is still debated in practice; the estimate above is the common choice


## Aside: theoretical versus practical REINFORCE

<table>
<thead><tr><th nowrap><div align="left">Feature</div></th><th nowrap><div align="left">Theoretical REINFORCE</div></th><th nowrap><div align="left">Practical (batch, GRPO-like)</div></th></tr></thead>
<tbody>
<tr><td nowrap><div align="left">rollouts per prompt</div></td><td nowrap><div align="left">typically 1</div></td><td nowrap><div align="left">several (for example $8$)</div></td></tr>
<tr><td nowrap><div align="left">baseline</div></td><td nowrap><div align="left">global moving average (historical)</div></td><td nowrap><div align="left">group mean (immediate)</div></td></tr>
<tr><td nowrap><div align="left">variance</div></td><td nowrap><div align="left">very high (small learning rates are usual)</div></td><td nowrap><div align="left">reduced by sampling more</div></td></tr>
<tr><td nowrap><div align="left">update frequency</div></td><td nowrap><div align="left">every trajectory</div></td><td nowrap><div align="left">after a group of trajectories</div></td></tr>
<tr><td nowrap><div align="left">compute per update</div></td><td nowrap><div align="left">low</div></td><td nowrap><div align="left">high (heavy generation)</div></td></tr>
<tr><td nowrap><div align="left">analogy</div></td><td nowrap><div align="left">your lifetime golf average</div></td><td nowrap><div align="left">several swings from the same spot</div></td></tr>
</tbody></table>


# What we established, and what it sets up

- **Established**
    - a baseline only needs to be independent of the action (the group mean is so only approximately)
    - a **group** of episodes from one start state can supply it, with no critic
    - the **group-relative advantage** is a completion's return minus the group mean, divided by the group standard deviation, copied to every token
    - **GRPO** is PPO's clipped objective with that advantage and a KL penalty to a reference model
        - the KL is a loss term here, where PPO puts it in the reward
    - the price: samples (generation), a noisy and slightly shrunk baseline, no per-step credit, and no signal when all rewards agree
    - **the finished slot table:** one theorem, three slots (weight, how far, baseline source)
- **Sets up**
    - **the limitations:** one weight for every token of a completion, and no signal when all rewards agree
        - the group baseline also needs one reward per completion: a verifier gives it, with no preference data
        - with no checker, a reward model learned from comparisons gives it (the gloss in C1)
    - the Advanced course: PPO and GRPO in depth and their refinements, reward models and RLHF, reasoning models
    - the Recap places all the methods of the series on one grid
